In [2]:
import pandas as pd
df = pd.read_csv("batdongsan_with_coordinates.csv", encoding="utf-8-sig")
df.head()

,Tên dự án,Giá,Diện tích,Phòng ngủ,Phòng vệ sinh,Vị trí,Định Vị,Latitude,Longitude
0,ĐỘC QUYỀN! QUỸ NGOẠI GIAO CĐT MELODY LINH ĐÀM ...,"3,6 tỷ",68 m²,2,2,"Hoàng Mai, Hà Nội","Đường Bằng Liệt, Phường Hoàng Liệt, Hoàng M...",20.966795,105.824457
1,TIN GIÁ THẬT. EM HIỆP CHUYÊN BÁN CĂN HỘ LUMIER...,"8,9 tỷ","74,2 m²",2,2,"Quận 2, Hồ Chí Minh","Đường Võ Nguyên Giáp, Phường An Phú, Quận 2...",10.736588,106.712024
2,"NHẬN NHÀ Ở HOẶC CHO THUÊ NGAY, CHIẾT KHẤU LÊN ...","2,72 tỷ","68,2 m²",2,2,"Dĩ An, Bình Dương","Đường An Bình, Phường An Bình, Dĩ An, Bình...",10.871419,106.755124
3,"ĐỘC QUYỀN 10 CĂN TẦNG TRUNG VIEW HỒ, STUDIO - ...",Giá thỏa thuận,60 m²,2,2,"Đông Anh, Hà Nội","Xã Đông Hội, Đông Anh, Hà Nội",21.139686,105.851855
4,"1PN-2PN-3PN VINHOMES Q9 CẮT LỖ 500-1.2 TỶ, CAM...","2,65 tỷ",60 m²,2,2,"Quận 9, Hồ Chí Minh","Phường Long Thạnh Mỹ, Quận 9, Hồ Chí Minh",10.758955,106.673049


In [3]:
!pip install osmnx geopy


     ---------------------------------------- 0.0/99.9 kB ? eta -:--:--
     ------------------------------------ --- 92.2/99.9 kB 2.6 MB/s eta 0:00:01
     ---------------------------------------- 99.9/99.9 kB 1.9 MB/s eta 0:00:00
  Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
     ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
     ----- ---------------------------------- 0.2/1.7 MB 4.6 MB/s eta 0:00:01
     ------ --------------------------------- 0.3/1.7 MB 3.4 MB/s eta 0:00:01
     ------- -------------------------------- 0.3/1.7 MB 2.4 MB/s eta 0:00:01
     ------- -------------------------------- 0.3/1.7 MB 2.4 MB/s eta 0:00:01
     ------- -------------------------------- 0.3/1.7 MB 2.4 MB/s eta 0:00:01
     -------- ------------------------------- 0.4/1.7 MB 1.3 MB/s eta 0:00:02
     --------- ------------------------------ 0.4/1.7 MB 1.3 MB/s eta 0:00:02
     --------- ------------------------------ 0.4/1.7 MB 1.3 MB/s eta 0:00:02
     --------- -----


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import osmnx as ox
from geopy.distance import geodesic

In [21]:
import requests
from math import radians, cos, sin, sqrt, atan2

# Hàm tính khoảng cách giữa hai tọa độ (Haversine formula)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Bán kính Trái Đất (km)
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c  # Khoảng cách (km)

# Hàm tìm các địa điểm xung quanh
def find_nearby_places(lat, lon, radius=3000):
    overpass_url = "http://overpass-api.de/api/interpreter"
    overpass_query = f"""
    [out:json];
    (
      node(around:{radius},{lat},{lon})["amenity"="hospital"];    // Bệnh viện
      node(around:{radius},{lat},{lon})["amenity"="school"];      // Trường học
      node(around:{radius},{lat},{lon})["amenity"="marketplace"]; // Chợ
      node(around:{radius},{lat},{lon})["shop"="mall"];           // Trung tâm thương mại
    );
    out body;
    """

    headers = {"User-Agent": "MyOSMApp/1.0 (your_email@example.com)"}
    response = requests.get(overpass_url, params={"data": overpass_query}, headers=headers)

    if response.status_code != 200:
        print("❌ Không thể truy vấn dữ liệu, thử lại sau!")
        return []

    data = response.json()
    places = []
    for element in data.get("elements", []):
        name = element.get("tags", {}).get("name", "Không có tên")
        place_lat = element["lat"]
        place_lon = element["lon"]
        distance = haversine(lat, lon, place_lat, place_lon)  # Tính khoảng cách
        places.append((name, distance))

    return sorted(places, key=lambda x: x[1])  # Sắp xếp theo khoảng cách

# Chạy hàm với tọa độ của bạn

# Thử với tọa độ đã có
latitude = 20.966795
longitude = 105.824457

results = find_nearby_places(latitude, longitude)
if results:
    for place in results:
        print(f"📍 {place[0]} - Khoảng cách: {place[1]:.2f} km")
else:
    print("⚠️ Không tìm thấy địa điểm nào trong phạm vi 3km.")


📍 Rice City Mall - Khoảng cách: 0.40 km
📍 Chợ dân sinh HH - Khoảng cách: 0.49 km
📍 Không có tên - Khoảng cách: 0.52 km
📍 Không có tên - Khoảng cách: 0.59 km
📍 Chợ Đại Từ - Khoảng cách: 0.93 km
📍 Chợ Xanh Linh Đàm - Khoảng cách: 1.02 km
📍 Trường THCS Tam Hiệp - Khoảng cách: 1.95 km
📍 Bệnh Viện Đa Khoa Thăng Long - Khoảng cách: 2.01 km
📍 Chợ Xanh Định Công - Khoảng cách: 2.23 km
📍 Hệ thống giáo dục May Academy - Khoảng cách: 2.33 km
📍 Trường THCS Thịnh Liệt - Khoảng cách: 2.40 km
📍 Trường THCS Lương Thế Vinh - Khoảng cách: 2.71 km
📍 Bệnh viện chuyên khoa Nam học và Hiếm muộn Việt - Bỉ - Khoảng cách: 2.75 km
📍 Trạm y tế Khương Hạ - Khoảng cách: 2.92 km
📍 Bệnh Viện Mắt Hồng Sơn - Khoảng cách: 2.94 km


Chạy thử với 1 dữ liệu

In [102]:
import requests
from math import radians, cos, sin, sqrt, atan2

# Hàm tính khoảng cách giữa hai tọa độ (Haversine formula)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Bán kính Trái Đất (km)
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c  # Khoảng cách (km)

# Hàm tìm số lượng tiện ích xung quanh và liệt kê từng loại kèm khoảng cách
def list_nearby_amenities(lat, lon, radius=2500):
    overpass_url = "http://overpass-api.de/api/interpreter"
    overpass_query = f"""
    [out:json];
    (
        node(around:{radius},{lat},{lon})["amenity"="hospital"];       // Trung tâm chăm sóc sức khỏe
      node(around:{radius},{lat},{lon})["amenity"="school"];         // Cơ sở giáo dục
      node(around:{radius},{lat},{lon})["amenity"="restaurant"];     // Cơ sở ăn uống
      node(around:{radius},{lat},{lon})["tourism"="attraction"];     // Địa điểm du lịch
      node(around:{radius},{lat},{lon})["public_transport"="stop"];  // Giao thông vận tải
    );
    out body;
    """

    headers = {"User-Agent": "MyOSMApp/1.0 (your_email@example.com)"}
    try:
        response = requests.get(overpass_url, params={"data": overpass_query}, headers=headers, timeout=10)
        response.raise_for_status()  # Kiểm tra mã trạng thái HTTP
    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi kết nối: {e}")
        return {}

    try:
        data = response.json()
    except ValueError:
        print("❌ Phản hồi không phải JSON hợp lệ. Có thể server không trả về dữ liệu.")
        return {}

    if "elements" not in data:
        print("⚠️ Không tìm thấy dữ liệu trong phản hồi.")
        return {}

    amenities = {
        "hospital": [],
        "school": [],
        "restaurant": [],
        "attraction": [],
        "public_transport": []
    }
    for element in data.get("elements", []):
        tags = element.get("tags", {})
        name = tags.get("name", "Không có tên")
        place_lat = element["lat"]
        place_lon = element["lon"]
        distance = haversine(lat, lon, place_lat, place_lon)  # Tính khoảng cách

        if tags.get("amenity") == "hospital":
            amenities["hospital"].append((name, distance))
        elif tags.get("amenity") == "school":
            amenities["school"].append((name, distance))
        elif tags.get("amenity") == "restaurant":
            amenities["restaurant"].append((name, distance))
        elif tags.get("tourism") == "attraction":
            amenities["attraction"].append((name, distance))
        elif tags.get("public_transport") == "stop":
            amenities["public_transport"].append((name, distance))

    return amenities

# Chạy hàm với tọa độ của bạn
latitude = 20.966795
longitude = 105.824457

results = list_nearby_amenities(latitude, longitude)
if results:
    print(f"📍 Các tiện ích xung quanh tọa độ ({latitude}, {longitude}):")
    for amenity_type, places in results.items():
        print(f"  - {amenity_type.capitalize()} ({len(places)}):")
        for name, distance in places:
            print(f"    + {name} - Khoảng cách: {distance:.2f} km")
else:
    print("⚠️ Không tìm thấy tiện ích nào trong phạm vi 2.5km.")

📍 Các tiện ích xung quanh tọa độ (20.966795, 105.824457):
  - Hospital (1):
    + Bệnh Viện Đa Khoa Thăng Long - Khoảng cách: 2.01 km
  - School (4):
    + Không có tên - Khoảng cách: 0.52 km
    + Trường THCS Thịnh Liệt - Khoảng cách: 2.40 km
    + Trường THCS Tam Hiệp - Khoảng cách: 1.95 km
    + Hệ thống giáo dục May Academy - Khoảng cách: 2.33 km
  - Restaurant (9):
    + Nhà Hàng Bia Hơi Hải Xồm - Khoảng cách: 1.82 km
    + nhà hàng tuấn vương - Khoảng cách: 0.72 km
    + Thảo Linh - Khoảng cách: 0.31 km
    + Nhà Hàng GoGi House - Khoảng cách: 0.39 km
    + Pizza Hut - Khoảng cách: 0.39 km
    + Nhà Hàng Kichi Kichi Linh Đàm - Khoảng cách: 0.40 km
    + Nhà hàng Phương Linh - Khoảng cách: 1.72 km
    + Quỳnh Quán - Khoảng cách: 0.87 km
    + Phở Việt 317 Lê Trọng Tấn - Khoảng cách: 2.37 km
  - Attraction (0):
  - Public_transport (0):


Tính tổng số lượng tiện ích và trung bình khoảng cách từ nhà đến các tiện ích này

In [ ]:
import pandas as pd
import requests
from math import radians, cos, sin, sqrt, atan2

# Hàm tính khoảng cách giữa hai tọa độ (Haversine formula)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Bán kính Trái Đất (km)
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c  # Khoảng cách (km)

# Hàm tìm các tiện ích xung quanh và tính tổng số lượng và khoảng cách trung bình
def calculate_amenities(lat, lon, radius=2500):
    overpass_url = "http://overpass-api.de/api/interpreter"
    overpass_query = f"""
    [out:json];
    (
      node(around:{radius},{lat},{lon})["amenity"="hospital"];
      node(around:{radius},{lat},{lon})["amenity"="school"];
      node(around:{radius},{lat},{lon})["amenity"="restaurant"];
      node(around:{radius},{lat},{lon})["tourism"="attraction"];
      node(around:{radius},{lat},{lon})["public_transport"="stop"];
    );
    out body;
    """

    headers = {"User-Agent": "MyOSMApp/1.0 (your_email@example.com)"}
    try:
        response = requests.get(overpass_url, params={"data": overpass_query}, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"❌ Lỗi kết nối: {e}")
        return {}, 0.0

    try:
        data = response.json()
    except ValueError:
        print("❌ Phản hồi không phải JSON hợp lệ.")
        return {}, 0.0

    if "elements" not in data:
        print("⚠️ Không tìm thấy dữ liệu trong phản hồi.")
        return {}, 0.0

    amenities = {
        "hospital": [],
        "school": [],
        "restaurant": [],
        "attraction": [],
        "public_transport": []
    }
    distances = []

    for element in data.get("elements", []):
        tags = element.get("tags", {})
        place_lat = element["lat"]
        place_lon = element["lon"]
        distance = haversine(lat, lon, place_lat, place_lon)
        distances.append(distance)

        if tags.get("amenity") == "hospital":
            amenities["hospital"].append(distance)
        elif tags.get("amenity") == "school":
            amenities["school"].append(distance)
        elif tags.get("amenity") == "restaurant":
            amenities["restaurant"].append(distance)
        elif tags.get("tourism") == "attraction":
            amenities["attraction"].append(distance)
        elif tags.get("public_transport") == "stop":
            amenities["public_transport"].append(distance)

    avg_distance_all = sum(distances) / len(distances) if distances else 0.0
    return amenities, avg_distance_all

# Đọc dữ liệu từ file CSV
df = pd.read_csv("batdongsan_with_coordinates.csv", encoding="utf-8-sig")

# Thêm các cột mới
df["Total_Amenities"] = 0
df["Average_Distance_All"] = 0.0
df["Average_Distance_Hospital"] = 0.0
df["Average_Distance_School"] = 0.0
df["Average_Distance_Restaurant"] = 0.0
df["Average_Distance_Attraction"] = 0.0
df["Average_Distance_Public_Transport"] = 0.0

# Lặp qua từng dòng để tính toán
for idx, row in df.iterrows():
    if pd.notna(row["Latitude"]) and pd.notna(row["Longitude"]):
        print(f"📍 Đang xử lý dòng {idx}...")
        amenities, avg_distance_all = calculate_amenities(row["Latitude"], row["Longitude"])
        
        # Tổng số lượng tiện ích
        total_amenities = sum(len(v) for v in amenities.values())
        df.at[idx, "Total_Amenities"] = total_amenities
        df.at[idx, "Average_Distance_All"] = avg_distance_all

        # Khoảng cách trung bình đến từng loại tiện ích
        df.at[idx, "Average_Distance_Hospital"] = sum(amenities["hospital"]) / len(amenities["hospital"]) if amenities["hospital"] else 0.0
        df.at[idx, "Average_Distance_School"] = sum(amenities["school"]) / len(amenities["school"]) if amenities["school"] else 0.0
        df.at[idx, "Average_Distance_Restaurant"] = sum(amenities["restaurant"]) / len(amenities["restaurant"]) if amenities["restaurant"] else 0.0
        df.at[idx, "Average_Distance_Attraction"] = sum(amenities["attraction"]) / len(amenities["attraction"]) if amenities["attraction"] else 0.0
        df.at[idx, "Average_Distance_Public_Transport"] = sum(amenities["public_transport"]) / len(amenities["public_transport"]) if amenities["public_transport"] else 0.0

# Lưu kết quả vào file mới
df.to_csv("batdongsan_with_amenity_features.csv", index=False, encoding="utf-8-sig")
print("✅ Đã hoàn thành và lưu vào 'batdongsan_with_amenity_features.csv'")

Thu thập dữ liệu giao thông